# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes
**Built on the position-tier baseline, not the Random Forest** — ML-09 found the baseline
ranks pages better (Spearman 0.166) than the model (0.058), so this playbook uses that proven
logic, extended into five archetypes instead of one flat "underperforming" flag.

**Archetypes → actions:**

| Archetype | Definition | Action |
|---|---|---|
| **Protect** | Good position (top_3/page_1/striking), CTR at/above its tier's benchmark | Leave alone — actively monitor for decline |
| **Improve** | Good position, CTR below benchmark, real volume (≥100 impressions) | Title/meta rewrite — the ML-07 queue |
| **Refresh candidate** | Weak position (page_3_5/deep) AND old (age ≥ 270 days, the paper's own decay-cliff threshold from Finding #2) | Content refresh, not just metadata — ties to the paper's Finding #4 freshness multiplier |
| **Monitor (low data)** | Impressions < 10 | Too little data to act on confidently (ML-08 showed ≤3-impression rows have 4.5x the prediction error of the rest) — wait for more data, don't act yet |
| **Low priority** | Weak position, low volume, not old enough for refresh logic | No action recommended — lowest-value segment |

**Reason codes:** `ctr_below_position_benchmark` (Improve), `aged_weak_position` (Refresh
candidate), `insufficient_data` (Monitor), `stable_or_outperforming` (Protect).

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd, numpy as np, os
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS impressions_total,
        ANY_VALUE(d.word_count) AS word_count,
        ANY_VALUE(d.content_type) AS content_type,
        DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS age_days
    FROM {FACT} f
    JOIN {DIM_CONTENT} d ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id, f.client_hash_id
""").df().dropna(subset=['ctr_observed', 'avg_position', 'age_days'])

def position_tier(p):
    if p <= 3: return '1_top_3'
    elif p <= 10: return '2_page_1'
    elif p <= 20: return '3_striking'
    elif p <= 50: return '4_page_3_5'
    else: return '5_deep'
data['position_tier'] = data['avg_position'].apply(position_tier)

benchmark = data.groupby('position_tier')['ctr_observed'].mean().rename('benchmark_ctr')
data = data.merge(benchmark, on='position_tier')
data['ctr_gap'] = data['benchmark_ctr'] - data['ctr_observed']
data['opportunity_score'] = (data['ctr_gap'] * data['impressions_total']).clip(lower=0)

def assign_archetype(row):
    good_position = row['position_tier'] in ['1_top_3', '2_page_1', '3_striking']
    weak_position = row['position_tier'] in ['4_page_3_5', '5_deep']
    if row['impressions_total'] < 10:
        return 'monitor_low_data', 'insufficient_data'
    if good_position and row['ctr_gap'] <= 0:
        return 'protect', 'stable_or_outperforming'
    if good_position and row['ctr_gap'] > 0 and row['impressions_total'] >= 100:
        return 'improve', 'ctr_below_position_benchmark'
    if weak_position and row['age_days'] >= 270:
        return 'refresh_candidate', 'aged_weak_position'
    return 'low_priority', 'below_action_threshold'

data[['archetype', 'reason_code']] = data.apply(lambda r: pd.Series(assign_archetype(r)), axis=1)

action_map = {
    'protect': 'monitor_only', 'improve': 'review_title_meta',
    'refresh_candidate': 'content_refresh', 'monitor_low_data': 'wait_for_data',
    'low_priority': 'no_action',
}
data['action_label'] = data['archetype'].map(action_map)

print("Archetype distribution:")
print(data['archetype'].value_counts())

queue = data[data['archetype'].isin(['improve', 'refresh_candidate'])].copy()
queue = queue.sort_values('opportunity_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)
cols = ['rank','content_hash_id','archetype','position_tier','ctr_observed','benchmark_ctr','age_days','impressions_total','opportunity_score','reason_code','action_label']
print(f"\nActionable queue (improve + refresh_candidate): {len(queue)} rows")
print(queue[cols].head(10).to_string())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Archetype distribution:
archetype
improve              61481
low_priority         51140
monitor_low_data     33465
protect              19147
refresh_candidate    11335
Name: count, dtype: int64

Actionable queue (improve + refresh_candidate): 72816 rows
   rank           content_hash_id archetype position_tier  ctr_observed  benchmark_ctr  age_days  impressions_total  opportunity_score                   reason_code       action_label
0     1  content_8d7d99f109e19aa2   improve       1_top_3      0.001420       0.012410       375           203497.0        2236.299802  ctr_below_position_benchmark  review_title_meta
1     2  content_0e03de7680314cd5   improve       1_top_3      0.003253       0.012410       375           221310.0        2026.350556  ctr_below_position_benchmark  review_title_meta
2     3  content_eadb33b5df496f4a   improve       1_top_3      0.009185       0.012410       375           617124.0        1990.211741  ctr_below_position_benchmark  review_title_meta
3     4  

**Intended use:** a prioritized starting list for a content/SEO team's weekly or monthly review
cycle — not an automated action system. A human reviews each flagged item before any change is
made. This is explicitly non-production: no code here writes to any live website or CMS.

**Who uses it:** a content strategist or SEO analyst deciding where to spend limited editing
time this week/month.

**Limits, stated plainly:**

1. **Single month, single snapshot.** All numbers come from March 2026 only. CTR is noisy
   month to month (ML-08 showed rows with ≤3 impressions have 4.5x the error of higher-volume
   rows) — a page flagged this month might look different next month purely from natural
   variance, not a real change.

2. **46 of 104 clients only** (per ML-04's data contract limitation) — clients without March
   GSC data are invisible to this playbook entirely. It cannot make recommendations for them.

3. **Baseline-driven, not model-driven, by design** — ML-09 showed the trained model actually
   ranks worse than this simpler position-tier logic. That's a deliberate, evidence-based
   choice, not a fallback — but it also means this playbook won't capture any pattern more
   complex than "CTR relative to position tier," even if such a pattern exists.

4. **`refresh_candidate` uses a threshold borrowed from the FlyRank research paper** (270-day
   decay cliff, Finding #2), not independently validated on this warehouse data. It's a
   reasonable starting assumption, not a proven cutoff for this specific dataset.

5. **No causal claim anywhere.** Nothing here claims that applying a recommended action will
   improve ranking or traffic — every recommendation is a correlational, decision-support
   suggestion based on observed patterns, consistent with the claim-language standard used
   throughout this capstone.

**Verified with real numbers:** 58 of 104 clients (56%) are entirely invisible to this
playbook — confirming limit #2 is not a minor edge case but affects the majority of clients.
19.0% of all rows fall below the 10-impression floor and get routed to `monitor_low_data`
rather than acted on. And 16.7% of all rows sit within 30 days of the 270-day refresh
threshold — meaning the `refresh_candidate` archetype is threshold-sensitive: a small,
reasonable change to that cutoff (say, 240 or 300 days instead of 270) would meaningfully
shift how many pages get flagged. This is exactly why limit #4 above says the threshold is
"a reasonable starting assumption, not a proven cutoff for this specific dataset."

In [4]:
# Verify the limits stated above with real numbers, not just assertions
total_clients = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_clients.parquet')").fetchone()[0]
clients_in_data = data['client_hash_id'].nunique()
print(f"Total clients in warehouse: {total_clients}")
print(f"Clients visible in this playbook: {clients_in_data}")
print(f"Clients this playbook cannot make recommendations for: {total_clients - clients_in_data}")

low_data_share = (data['impressions_total'] < 10).mean()
print(f"\nShare of all rows below the 10-impression 'monitor_low_data' floor: {low_data_share:.1%}")

# Sensitivity check on the refresh_candidate age threshold (270 days, borrowed from the paper)
near_boundary = data[(data['age_days'] >= 240) & (data['age_days'] <= 300)]
print(f"\nRows within +/-30 days of the 270-day refresh threshold: {len(near_boundary)} "
      f"({len(near_boundary)/len(data):.1%} of all rows) -- this many rows sitting close to the "
      f"cutoff means small threshold changes could meaningfully shift the refresh_candidate count.")


Total clients in warehouse: 104
Clients visible in this playbook: 46
Clients this playbook cannot make recommendations for: 58

Share of all rows below the 10-impression 'monitor_low_data' floor: 19.0%

Rows within +/-30 days of the 270-day refresh threshold: 29558 (16.7% of all rows) -- this many rows sitting close to the cutoff means small threshold changes could meaningfully shift the refresh_candidate count.


## 3. Human review + the no-go list

**What a human must check before acting on any flagged item:**

- **Query intent** — is this page ranking for a navigational/branded query? Low CTR there may
  be normal (searchers already know the destination), not a title/meta problem (same caution
  raised in ML-07's top-20 review).
- **SERP layout** — does this query's results page have heavy ads, shopping results, or a
  featured snippet already answering the question? Organic CTR is structurally suppressed
  there regardless of title quality.
- **Cannibalization** — does the same client have another page targeting a very similar query?
  A rewrite might just shift clicks between two of their own pages rather than create new ones.
- **Business priority** — opportunity_score reflects traffic volume, not business value; a
  high-margin product page may deserve attention before a higher-scoring blog post.

**What should NEVER be automated here:**

- **Auto-publishing any title/meta rewrite.** This system flags candidates; it does not draft
  or ship copy. A human writer/editor makes the actual content decision.
- **Auto-deleting or auto-merging pages**, even ones that would score as classic "prune"
  candidates (very old, near-zero impressions). Removing a page is a one-way, high-risk action
  that depends on business context this data cannot see (legal pages, seasonal content, internal
  links from other important pages).
- **Treating `opportunity_score` as a KPI target.** Optimizing to move this specific number
  risks gaming metadata for the score rather than genuinely improving the page for readers.
- **Any action based on a single month's data alone** — per the limits above, one month is too
  noisy a signal to justify an irreversible action on its own.

**Confirmed with real numbers:** 101 client/position-tier combinations have 5+ flagged pages
each — and the top case (one client, page_1 tier) has **10,073** flagged pages simultaneously.
At that scale, treating each row as an independent recommendation is clearly wrong — a single
client-level review (is this an unusually large site? a templated page pattern? one systemic
title issue across thousands of pages?) would be far more useful than 10,073 individual
one-off edits. This confirms the cannibalization/context caution above isn't a theoretical
edge case — it's a real pattern affecting a meaningful share of the queue, and a large-client
rollup view should sit above the raw ranked list in any real workflow.

In [5]:
cannibalization_check = queue.groupby(['client_hash_id', 'position_tier']).size().reset_index(name='n_flagged_same_tier')
risky = cannibalization_check[cannibalization_check['n_flagged_same_tier'] >= 5].sort_values('n_flagged_same_tier', ascending=False)
print(f"Client x position-tier groups with 5+ flagged pages (worth a manual cannibalization check): {len(risky)}")
print(risky.head(10).to_string(index=False))


Client x position-tier groups with 5+ flagged pages (worth a manual cannibalization check): 101
         client_hash_id position_tier  n_flagged_same_tier
client_73cda7b4e4f265ea      2_page_1                10073
client_62f4a7e64f5e0096      2_page_1                 8540
client_73cda7b4e4f265ea       1_top_3                 3071
client_73cda7b4e4f265ea    4_page_3_5                 2938
client_73cda7b4e4f265ea    3_striking                 2868
client_e547b89c05043229      2_page_1                 2810
client_62f4a7e64f5e0096       1_top_3                 2774
client_23a62021009f63c4    3_striking                 2383
client_fef1a8f436438636      2_page_1                 2376
client_08a6a72ff48e62c0      2_page_1                 2359


## 4. Monitoring / retrain triggers

**When would this playbook go stale, and how would you know?**

- **Monthly refresh, minimum.** Rebuild the queue every month as new GSC data lands — a single
  month's snapshot (per Section 2's limits) shouldn't be trusted for more than one review cycle.

- **Retrain trigger 1 — benchmark drift.** If the position-tier benchmark CTRs (the core of the
  `improve` and `protect` logic) shift by more than ~20% month-over-month for any tier, that
  signals something changed in the search landscape itself (a Google update, a seasonal shift),
  not just noise — worth a manual review of the whole approach before trusting the next queue.

- **Retrain trigger 2 — archetype distribution shift.** If the share of pages in `monitor_low_data`
  changes sharply (e.g. jumps from ~19% to 35%+), that suggests a tracking or data-pipeline issue
  upstream, not a real change in the portfolio — worth investigating before acting on that
  month's queue at all.

- **Retrain trigger 3 — client coverage change.** If the 46-of-104 client coverage (Section 2)
  changes significantly, the client population this playbook represents has shifted, and past
  performance patterns may not transfer to the newly-included or newly-excluded clients.

- **What this is NOT:** a real-time or continuously-retrained system. Given the non-production,
  human-reviewed intent (Section 2), monthly batch rebuilding is appropriate — anything faster
  would outpace the human review step meant to sit between recommendation and action.
  **Concrete thresholds for Trigger 1** (computed above, from this March 2026 run): a future
month's benchmark should trigger a manual review if it falls outside these ranges —
top_3: 0.0099–0.0149, page_1: 0.0039–0.0059, striking: 0.0026–0.0039, page_3_5: 0.0018–0.0027,
deep: 0.0007–0.0011. These aren't hardcoded forever — recompute and update them at each
monthly rebuild, using this month's number as the new center point.

In [6]:
# Show the actual current benchmark values, so the 20%-drift trigger has a concrete baseline to compare future months against
current_benchmarks = data.groupby('position_tier')['ctr_observed'].agg(['mean', 'size']).rename(columns={'mean': 'benchmark_ctr', 'size': 'n'})
current_benchmarks['drift_alert_below'] = (current_benchmarks['benchmark_ctr'] * 0.8).round(6)
current_benchmarks['drift_alert_above'] = (current_benchmarks['benchmark_ctr'] * 1.2).round(6)
print("March 2026 benchmarks and the +/-20% drift-alert thresholds for future months:")
print(current_benchmarks)


March 2026 benchmarks and the +/-20% drift-alert thresholds for future months:
               benchmark_ctr      n  drift_alert_below  drift_alert_above
position_tier                                                            
1_top_3             0.012410  17560           0.009928           0.014891
2_page_1            0.004931  81856           0.003945           0.005918
3_striking          0.003211  32195           0.002569           0.003853
4_page_3_5          0.002288  33281           0.001830           0.002745
5_deep              0.000904  11676           0.000723           0.001084


## 5. Exports for the paper
Exporting the ranked queue (kept out of git, per the CI leak-guard — this notebook regenerates
it on demand) plus the archetype distribution and key metrics as committed JSON receipts, and
the position-tier benchmark chart as a committed figure — these three are what next week's
paper will pull numbers and visuals from directly.

In [7]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. The ranked queue CSV (stays OUT of git per CI leak-guard — regenerated each run)
queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"Wrote work/outputs/action_playbook_queue.csv ({len(queue)} rows)")

# 2. Metrics JSON — the receipts, safe to commit (aggregates only, no raw rows)
playbook_metrics = {
    'total_pages_analyzed': int(len(data)),
    'archetype_distribution': data['archetype'].value_counts().to_dict(),
    'actionable_queue_size': int(len(queue)),
    'clients_visible': int(data['client_hash_id'].nunique()),
    'clients_total_in_warehouse': int(total_clients),
    'position_tier_benchmarks': current_benchmarks['benchmark_ctr'].to_dict(),
    'cannibalization_risk_groups': int(len(risky)),
    'source_month': '2026-03',
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(playbook_metrics, f, indent=2, default=str)
print("Wrote work/outputs/playbook_metrics.json")

# 3. The benchmark chart, committed as a figure for the paper
plt.figure(figsize=(7, 5))
tiers_ordered = ['1_top_3', '2_page_1', '3_striking', '4_page_3_5', '5_deep']
plt.bar(tiers_ordered, current_benchmarks.loc[tiers_ordered, 'benchmark_ctr'])
plt.ylabel('Benchmark CTR')
plt.title('March 2026 CTR Benchmark by Position Tier')
plt.tight_layout()
plt.savefig('work/figures/position_tier_benchmark.png', dpi=150)
plt.close()
print("Wrote work/figures/position_tier_benchmark.png")

print("\nAll exports complete.")

Wrote work/outputs/action_playbook_queue.csv (72816 rows)
Wrote work/outputs/playbook_metrics.json
Wrote work/figures/position_tier_benchmark.png

All exports complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.